# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [1]:
import json
from pathlib import Path

import pandas as pd

noticias = []

for arquivo in sorted(Path("../WebScrapping/data/").glob("*.json")):
    with arquivo.open(encoding="utf-8") as f:
        noticias.append(json.load(f))

df = pd.DataFrame(noticias)

print(f"Antes: {len(df)} noticias")

df = df.drop_duplicates(subset="url").reset_index(drop=True)

print(f"Depois: {len(df)} noticias")

df.head()

Antes: 1300 noticias
Depois: 1297 noticias


,url,titulo,subtitulo,categoria,data_publicacao,imagem,corpo
0,https://www.cnnbrasil.com.br/esportes/futebol/...,Santos acerta renovação de Robinho Jr. até o f...,Clube da Baixada Santista define condições de ...,Santos,2026-04-14T16:18:41-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,O Santos encaminhou a extensão de contrato do ...
1,https://www.cnnbrasil.com.br/esportes/futebol/...,Cruzeiro anuncia renovação com Artur Jorge apó...,Treinador iniciou no clube celeste em 22 de ma...,Cruzeiro,2026-04-14T15:52:24-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,"O Cruzeiro anunciou, nesta terça-feira (14), a..."
2,https://www.cnnbrasil.com.br/esportes/futebol/...,Ex-Corinthians cita caso vivido em clube e cri...,Betão comentou o episódio recente no Dérbi pau...,Corinthians,2026-04-13T18:37:05-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,Um episódio lamentável de racismo marcou o clá...
3,https://www.cnnbrasil.com.br/esportes/futebol/...,Cruzeiro: Pedro Junio explica escolha por Artu...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",Cruzeiro,2026-04-10T09:00:53-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Pedro Junio, vice-presidente do Cruzeiro, expl..."
4,https://www.cnnbrasil.com.br/esportes/futebol/...,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Treinador destaca evolução da equipe e comenta...,Flamengo,2026-03-12T08:46:42-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,Após a vitória do Flamengo sobre o Cruzeiro po...


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [2]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["corpo"].apply(limpar_texto)

df[["corpo", "texto_limpo"]].head()

,corpo,texto_limpo
0,O Santos encaminhou a extensão de contrato do ...,o santos encaminhou a extensao de contrato do ...
1,"O Cruzeiro anunciou, nesta terça-feira (14), a...",o cruzeiro anunciou nesta terca feira 14 a ren...
2,Um episódio lamentável de racismo marcou o clá...,um episodio lamentavel de racismo marcou o cla...
3,"Pedro Junio, vice-presidente do Cruzeiro, expl...",pedro junio vice presidente do cruzeiro explic...
4,Após a vitória do Flamengo sobre o Cruzeiro po...,apos a vitoria do flamengo sobre o cruzeiro po...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [3]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,o santos encaminhou a extensao de contrato do ...,"[santos, encaminhou, extensao, contrato, ataca...",santos encaminhou extensao contrato atacante r...
1,o cruzeiro anunciou nesta terca feira 14 a ren...,"[cruzeiro, anunciou, nesta, terca, feira, 14, ...",cruzeiro anunciou nesta terca feira 14 renovac...
2,um episodio lamentavel de racismo marcou o cla...,"[episodio, lamentavel, racismo, marcou, classi...",episodio lamentavel racismo marcou classico co...
3,pedro junio vice presidente do cruzeiro explic...,"[pedro, junio, vice, presidente, cruzeiro, exp...",pedro junio vice presidente cruzeiro explicou ...
4,apos a vitoria do flamengo sobre o cruzeiro po...,"[apos, vitoria, flamengo, sobre, cruzeiro, 2, ...",apos vitoria flamengo sobre cruzeiro 2 0 quint...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,01,0100,01a,02,03,031,034,038,...,zortea,zorz,zrndj7qcoo,zubeldia,zudcuxlcuk,zugvs0nt0tg,zulu,zvezda,zvglllrzo1,zygluzo8cn0
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [5]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

1003 colunas removidas


,aa,aab,abafa,abaixa,abaixar,abaixo,abaixou,abala,abalada,abalado,...,zoando,zoeira,zona,zonas,zortea,zorz,zubeldia,zudcuxlcuk,zulu,zvezda
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [6]:
metadados = df[["data_publicacao", "titulo", "categoria", "url", "imagem", "subtitulo"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data_publicacao,titulo,categoria,url,imagem,subtitulo,bow_aa,bow_aab,bow_abafa,bow_abaixa,...,bow_zoando,bow_zoeira,bow_zona,bow_zonas,bow_zortea,bow_zorz,bow_zubeldia,bow_zudcuxlcuk,bow_zulu,bow_zvezda
0,2026-04-14T16:18:41-03:00,Santos acerta renovação de Robinho Jr. até o f...,Santos,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Clube da Baixada Santista define condições de ...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2026-04-14T15:52:24-03:00,Cruzeiro anuncia renovação com Artur Jorge apó...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador iniciou no clube celeste em 22 de ma...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2026-04-13T18:37:05-03:00,Ex-Corinthians cita caso vivido em clube e cri...,Corinthians,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Betão comentou o episódio recente no Dérbi pau...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2026-04-10T09:00:53-03:00,Cruzeiro: Pedro Junio explica escolha por Artu...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
4,2026-03-12T08:46:42-03:00,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Flamengo,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador destaca evolução da equipe e comenta...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [7]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 15785


bow_clube          1381
bow_apos           1246
bow_copa           1193
bow_feira          1139
bow_jogo           1120
bow_contra         1086
bow_campeonato     1063
bow_corinthians    1033
bow_brasil         1019
bow_tecnico        1014
dtype: int64

In [8]:
frequencia_palavras.tail(10)

bow_moderna       1
bow_money         1
bow_mondragon     1
bow_momentanea    1
bow_abaixa        1
bow_abalou        1
bow_abandono      1
bow_abecat        1
bow_zajkov        1
bow_zudcuxlcuk    1
dtype: int64

In [9]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

feira         766
apos          694
brasileiro    657
contra        645
campeonato    644
jogo          613
clube         613
tecnico       586
copa          558
nesta         554
dtype: int64

In [10]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
457,Corinthians anuncia ex-goleiro como novo geren...,39
640,Veja gol de Igor Thiago para o Brasil na vitór...,42
737,Flamengo: Bap diz que não vai construir estádio,42
472,Pedro iguala Gabigol como artilheiro do Flamen...,45
467,Jhon Arias faz golaço para o Palmeiras contra ...,46
232,Tirol homenageia América-MG antes de duelo pel...,48
856,Paysandu x GAS: horário e onde assistir à Copa...,49
566,Neymar dá passe para gol do Santos; veja,51
1037,Vasco x Fluminense: Canobbio faz gol mais rápi...,51
477,Lautaro Díaz marca golaço contra o Flamengo no...,52


## 4. TF-IDF

O Bag of Words transforma textos em numeros contando quantas vezes cada palavra aparece em cada noticia. Essa representacao e simples e util, mas tem uma limitacao importante: palavras muito frequentes podem parecer importantes apenas porque aparecem muito.

O **TF-IDF** e uma forma de dar peso para as palavras tentando responder a duas perguntas ao mesmo tempo:

- Esta palavra aparece bastante nesta noticia?
- Esta palavra e rara no conjunto de noticias?

A ideia e que uma palavra deve receber peso alto quando aparece bastante em uma noticia, mas nao aparece em todas as noticias. Assim, o TF-IDF ajuda a destacar palavras mais caracteristicas de cada documento.

### TF: Term Frequency

**TF** significa *Term Frequency*, ou frequencia do termo.

Ele mede a importancia de uma palavra dentro de uma noticia especifica. Se uma palavra aparece muitas vezes em uma noticia, o TF dela naquela noticia sera maior.

$$TF(t, d) = \frac{\text{quantidade do termo } t \text{ no documento } d}{\text{total de termos no documento } d}$$

Exemplo: se uma noticia tem 100 palavras e a palavra `senado` aparece 5 vezes, entao:

$$TF(\text{senado}) = \frac{5}{100} = 0.05$$

O TF olha apenas para dentro de uma noticia. Ele ainda nao sabe se a palavra e comum ou rara no conjunto inteiro.

### IDF: Inverse Document Frequency

**IDF** significa *Inverse Document Frequency*, ou frequencia inversa nos documentos.

Ele mede o quanto uma palavra e rara no conjunto de noticias. Palavras que aparecem em muitas noticias recebem peso menor. Palavras que aparecem em poucas noticias recebem peso maior.

$$IDF(t) = \log\left(\frac{N}{DF(t)}\right)$$

Onde:

- $N$ e o numero total de noticias.
- $DF(t)$ e o numero de noticias em que o termo $t$ aparece.

Se uma palavra aparece em todas as noticias, ela nao ajuda muito a diferenciar uma noticia da outra. Por isso, seu IDF fica baixo.

### TF-IDF

O **TF-IDF** combina as duas ideias:

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

Uma palavra tera peso alto quando:

- aparece bastante em uma noticia;
- nao aparece em muitas outras noticias.

Por isso, TF-IDF costuma ser melhor do que uma contagem simples quando queremos comparar documentos, encontrar palavras importantes ou preparar textos para modelos de aprendizado de maquina.

## Como calcular TF-IDF na mao

Como ja criamos o `df_final`, temos uma matriz Bag of Words pronta nas colunas que comecam com `bow_`.

Para calcular o TF-IDF manualmente com pandas, o caminho e:

1. Separar apenas as colunas `bow_` do `df_final`.
2. Remover o prefixo `bow_` dos nomes das colunas, para trabalhar diretamente com as palavras.
3. Calcular o total de palavras de cada noticia somando as linhas da matriz.
4. Calcular o **TF** dividindo cada valor da linha pelo total da propria linha.
5. Calcular em quantas noticias cada palavra aparece.
6. Calcular o **IDF** usando $\log(N / DF)$.
7. Multiplicar a matriz de TF pelo vetor de IDF.

O resultado final e uma nova matriz: cada linha continua sendo uma noticia, cada coluna continua sendo uma palavra, mas agora os valores sao pesos TF-IDF em vez de contagens brutas.

## Implementacao

Agora vamos implementar. A ideia e preencher as proximas celulas seguindo os passos explicados acima.

In [11]:
matriz_bow_df = df_final[colunas_bow].copy()
matriz_bow_df.columns = matriz_bow_df.columns.str.replace("bow_", "", regex=False)

matriz_bow_df.head()

,aa,aab,abafa,abaixa,abaixar,abaixo,abaixou,abala,abalada,abalado,...,zoando,zoeira,zona,zonas,zortea,zorz,zubeldia,zudcuxlcuk,zulu,zvezda
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
total_palavras_por_noticia = matriz_bow_df.sum(axis=1)

matriz_tf = matriz_bow_df.div(total_palavras_por_noticia, axis=0)

matriz_tf.head()

,aa,aab,abafa,abaixa,abaixar,abaixo,abaixou,abala,abalada,abalado,...,zoando,zoeira,zona,zonas,zortea,zorz,zubeldia,zudcuxlcuk,zulu,zvezda
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.014851,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
noticia = 0

print(df_final.loc[noticia, "titulo"])

matriz_tf.loc[noticia].sort_values(ascending=False).head(10)

Santos acerta renovação de Robinho Jr. até o fim de 2031; veja detalhes


atleta          0.018519
dezembro        0.018519
desempenho      0.018519
ainda           0.018519
clube           0.018519
jogador         0.018519
novo            0.018519
profissional    0.018519
santos          0.018519
time            0.018519
Name: 0, dtype: float64

In [14]:
import numpy as np

N = len(matriz_bow_df)
documentos_por_termo = (matriz_bow_df > 0).sum(axis=0)

idf = np.log(N / documentos_por_termo)

df_idf = pd.DataFrame({
    "documentos_com_o_termo": documentos_por_termo,
    "idf": idf
}).sort_values("idf", ascending=False)

df_idf.head(10)

,documentos_com_o_termo,idf
zvezda,1,7.167809
zudcuxlcuk,1,7.167809
aab,1,7.167809
zorz,1,7.167809
abaixa,1,7.167809
zenon,1,7.167809
zegarra,1,7.167809
zanotti,1,7.167809
zamora,1,7.167809
abordagens,1,7.167809


In [15]:
df_idf.tail(10)

,documentos_com_o_termo,idf
nesta,554,0.850644
copa,558,0.843450
tecnico,586,0.794489
jogo,613,0.749444
clube,613,0.749444
campeonato,644,0.700110
contra,645,0.698559
brasileiro,657,0.680125
apos,694,0.625337
feira,766,0.526627


In [16]:
matriz_tfidf = matriz_tf.mul(idf, axis=1)

matriz_tfidf.head()

,aa,aab,abafa,abaixa,abaixar,abaixo,abaixou,abala,abalada,abalado,...,zoando,zoeira,zona,zonas,zortea,zorz,zubeldia,zudcuxlcuk,zulu,zvezda
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.041189,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
noticia = 10

print(df_final.loc[noticia, "titulo"])

matriz_tfidf.loc[noticia].sort_values(ascending=False).head(10)

Pai de Gerson, do Cruzeiro, reage a ataques de torcedores do Flamengo


marcao           0.079449
hostilizado      0.056961
gerson           0.052171
cruzeiro         0.046386
scores           0.035309
vacilos          0.035309
ressentimento    0.035309
replay           0.035309
vez              0.034870
intervalo        0.032686
Name: 10, dtype: float64

In [18]:
tfidf_com_prefixo = matriz_tfidf.add_prefix("tfidf_").reset_index(drop=True)

df_final_tfidf = pd.concat([metadados, tfidf_com_prefixo], axis=1)

df_final_tfidf.head()

,data_publicacao,titulo,categoria,url,imagem,subtitulo,tfidf_aa,tfidf_aab,tfidf_abafa,tfidf_abaixa,...,tfidf_zoando,tfidf_zoeira,tfidf_zona,tfidf_zonas,tfidf_zortea,tfidf_zorz,tfidf_zubeldia,tfidf_zudcuxlcuk,tfidf_zulu,tfidf_zvezda
0,2026-04-14T16:18:41-03:00,Santos acerta renovação de Robinho Jr. até o f...,Santos,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Clube da Baixada Santista define condições de ...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-04-14T15:52:24-03:00,Cruzeiro anuncia renovação com Artur Jorge apó...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador iniciou no clube celeste em 22 de ma...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-04-13T18:37:05-03:00,Ex-Corinthians cita caso vivido em clube e cri...,Corinthians,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Betão comentou o episódio recente no Dérbi pau...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-04-10T09:00:53-03:00,Cruzeiro: Pedro Junio explica escolha por Artu...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",0.0,0.0,0.0,0.0,...,0.0,0.0,0.041189,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-03-12T08:46:42-03:00,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Flamengo,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador destaca evolução da equipe e comenta...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Como calcular TF-IDF com scikit-learn

Na pratica, tambem podemos usar o `TfidfVectorizer` do scikit-learn.

Ele faz em uma unica etapa o que fizemos manualmente:

- cria o vocabulario;
- conta os termos;
- calcula os pesos TF-IDF;
- devolve uma matriz numerica.

Como nossos textos ja foram limpos anteriormente, podemos aplicar o `TfidfVectorizer` diretamente em `df["texto_sem_stopwords"]`.

A vantagem do scikit-learn e que ele ja resolve detalhes de implementacao e devolve uma matriz pronta para ser usada em modelos de aprendizado de maquina. A vantagem de fazer na mao primeiro e entender o que esta acontecendo por baixo.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer()
matriz_tfidf_sklearn = vectorizer_tfidf.fit_transform(df["texto_sem_stopwords"])

df_tfidf_sklearn = pd.DataFrame(
    matriz_tfidf_sklearn.toarray(),
    columns=vectorizer_tfidf.get_feature_names_out()
)

df_tfidf_sklearn.head()

,00,000,01,0100,01a,02,03,031,034,038,...,zortea,zorz,zrndj7qcoo,zubeldia,zudcuxlcuk,zugvs0nt0tg,zulu,zvezda,zvglllrzo1,zygluzo8cn0
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Salvando os dados

Vamos salvar as matrizes Bag of Words e TF-IDF em arquivos CSV para reaproveitar em outras analises (por exemplo, visualizacao com t-SNE).

In [20]:
df.head()

,url,titulo,subtitulo,categoria,data_publicacao,imagem,corpo,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,https://www.cnnbrasil.com.br/esportes/futebol/...,Santos acerta renovação de Robinho Jr. até o f...,Clube da Baixada Santista define condições de ...,Santos,2026-04-14T16:18:41-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,O Santos encaminhou a extensão de contrato do ...,o santos encaminhou a extensao de contrato do ...,"[santos, encaminhou, extensao, contrato, ataca...",santos encaminhou extensao contrato atacante r...
1,https://www.cnnbrasil.com.br/esportes/futebol/...,Cruzeiro anuncia renovação com Artur Jorge apó...,Treinador iniciou no clube celeste em 22 de ma...,Cruzeiro,2026-04-14T15:52:24-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,"O Cruzeiro anunciou, nesta terça-feira (14), a...",o cruzeiro anunciou nesta terca feira 14 a ren...,"[cruzeiro, anunciou, nesta, terca, feira, 14, ...",cruzeiro anunciou nesta terca feira 14 renovac...
2,https://www.cnnbrasil.com.br/esportes/futebol/...,Ex-Corinthians cita caso vivido em clube e cri...,Betão comentou o episódio recente no Dérbi pau...,Corinthians,2026-04-13T18:37:05-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,Um episódio lamentável de racismo marcou o clá...,um episodio lamentavel de racismo marcou o cla...,"[episodio, lamentavel, racismo, marcou, classi...",episodio lamentavel racismo marcou classico co...
3,https://www.cnnbrasil.com.br/esportes/futebol/...,Cruzeiro: Pedro Junio explica escolha por Artu...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",Cruzeiro,2026-04-10T09:00:53-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Pedro Junio, vice-presidente do Cruzeiro, expl...",pedro junio vice presidente do cruzeiro explic...,"[pedro, junio, vice, presidente, cruzeiro, exp...",pedro junio vice presidente cruzeiro explicou ...
4,https://www.cnnbrasil.com.br/esportes/futebol/...,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Treinador destaca evolução da equipe e comenta...,Flamengo,2026-03-12T08:46:42-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,Após a vitória do Flamengo sobre o Cruzeiro po...,apos a vitoria do flamengo sobre o cruzeiro po...,"[apos, vitoria, flamengo, sobre, cruzeiro, 2, ...",apos vitoria flamengo sobre cruzeiro 2 0 quint...


In [21]:
from pathlib import Path

pasta_dados = Path("../dados")
pasta_dados.mkdir(exist_ok=True)

# Dados completos: metadados + texto original + texto limpo + texto sem stopwords
df_completo = df[["url", "titulo", "subtitulo", "categoria", "data_publicacao", "imagem", "texto_limpo","corpo", "texto_sem_stopwords"]]
df_completo.to_csv(pasta_dados / "noticias.csv", index=False)

# Bag of Words (metadados + colunas bow_)
df_final.to_csv(pasta_dados / "bow.csv", index=False)

# TF-IDF (metadados + colunas tfidf_)
df_final_tfidf.to_csv(pasta_dados / "tfidf.csv", index=False)

print(f"Noticias completas: {pasta_dados / 'noticias.csv'} (shape={df_completo.shape})")
print(f"Bag of Words:       {pasta_dados / 'bow.csv'}      (shape={df_final.shape})")
print(f"TF-IDF:             {pasta_dados / 'tfidf.csv'}    (shape={df_final_tfidf.shape})")

Noticias completas: ..\dados\noticias.csv (shape=(1297, 9))
Bag of Words:       ..\dados\bow.csv      (shape=(1297, 15792))
TF-IDF:             ..\dados\tfidf.csv    (shape=(1297, 15791))
